# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/rambo/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-07-24 17:55:49.018268: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-24 17:55:49.066174: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-24 17:55:49.301984: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-24 17:55:49.302022: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-24 17:55:49.303483: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [8]:
# Step 1: Separate features and labels

np.random.seed(42)
tf.random.set_seed(42)

X = df.drop(columns=["Class"]).to_numpy(dtype=np.float32)
y = df["Class"].to_numpy(dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", np.unique(y))

X shape: (178, 13)
y shape: (178,)
Classes: [0 1 2]


In [9]:
# Step 2: 70/30 train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 124
Testing samples: 54


In [10]:
# Step 3: Standardize features

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print(
    "Mean of scaled training features:",
    np.round(X_train_scaled.mean(axis=0), 4),
)

print(
    "Standard deviation of scaled training features:",
    np.round(X_train_scaled.std(axis=0), 4),
)

Mean of scaled training features: [ 0. -0.  0. -0.  0. -0.  0.  0.  0.  0.  0. -0.  0.]
Standard deviation of scaled training features: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [11]:
# Step 4: One-hot encode labels

y_train_cat = to_categorical(
    y_train,
    num_classes=num_classes,
)

y_test_cat = to_categorical(
    y_test,
    num_classes=num_classes,
)

print("Training-label shape:", y_train_cat.shape)
print("Testing-label shape:", y_test_cat.shape)

Training-label shape: (124, 3)
Testing-label shape: (54, 3)


In [12]:
# Step 5: Define the base DNN

model = Sequential([
    tf.keras.Input(shape=(num_features,)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax"),
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
# Step 6: Compile and train

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

base_history = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1,
)

Epoch 1/20
13/13 [==============================] - 0s 10ms/step - loss: 0.0109 - accuracy: 1.0000 - val_loss: 0.0295 - val_accuracy: 1.0000
Epoch 2/20
13/13 [==============================] - 0s 4ms/step - loss: 0.0055 - accuracy: 1.0000 - val_loss: 0.0242 - val_accuracy: 1.0000
Epoch 3/20
13/13 [==============================] - 0s 4ms/step - loss: 0.0036 - accuracy: 1.0000 - val_loss: 0.0209 - val_accuracy: 1.0000
Epoch 4/20
13/13 [==============================] - 0s 3ms/step - loss: 0.0024 - accuracy: 1.0000 - val_loss: 0.0243 - val_accuracy: 1.0000
Epoch 5/20
13/13 [==============================] - 0s 4ms/step - loss: 0.0018 - accuracy: 1.0000 - val_loss: 0.0277 - val_accuracy: 1.0000
Epoch 6/20
13/13 [==============================] - 0s 3ms/step - loss: 0.0013 - accuracy: 1.0000 - val_loss: 0.0265 - val_accuracy: 1.0000
Epoch 7/20
13/13 [==============================] - 0s 3ms/step - loss: 0.0011 - accuracy: 1.0000 - val_loss: 0.0239 - val_accuracy: 1.0000
Epoch 8/20
13/13 [=

In [17]:
# Step 7: Evaluate the base model

base_test_loss, base_test_accuracy = model.evaluate(
    X_test_scaled,
    y_test_cat,
    verbose=0,
)

base_probabilities = model.predict(
    X_test_scaled,
    verbose=0,
)

base_predictions = np.argmax(
    base_probabilities,
    axis=1,
)

print(f"Test loss: {base_test_loss:.4f}")
print(f"Test accuracy: {base_test_accuracy:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        base_predictions,
        labels=np.arange(num_classes),
        target_names=wine.target_names,
        zero_division=0,
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_test,
        base_predictions,
        labels=np.arange(num_classes),
    )
)

Test loss: 0.0578
Test accuracy: 0.9815

Classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


In [19]:
# Step 8: Convert the base model to TFLite

import os


def file_size_kb(filename):
    return os.path.getsize(filename) / 1024.0


base_converter = tf.lite.TFLiteConverter.from_keras_model(model)
base_tflite_model = base_converter.convert()

with open("model_base.tflite", "wb") as file:
    file.write(base_tflite_model)

base_tflite_size_kb = file_size_kb("model_base.tflite")

print(
    f"Base TFLite model size: "
    f"{base_tflite_size_kb:.2f} KB"
)

INFO:tensorflow:Assets written to: /tmp/tmpq0lj19wy/assets


INFO:tensorflow:Assets written to: /tmp/tmpq0lj19wy/assets


Base TFLite model size: 14.06 KB


2026-07-24 18:23:50.540976: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:23:50.541052: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:23:50.541273: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpq0lj19wy
2026-07-24 18:23:50.541958: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:23:50.541968: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpq0lj19wy
2026-07-24 18:23:50.544103: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:23:50.583466: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpq0lj19wy
2026-07-24 18:23:50.590854: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 49581 m

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [20]:
def representative_data_gen(X_reference, num_samples=100):
    max_samples = min(num_samples, len(X_reference))

    for index in range(max_samples):
        sample = X_reference[index:index + 1]
        yield [sample.astype(np.float32)]


def quantize_and_evaluate(
    model,
    X_test,
    y_test_cat,
    quant_type,
    filename,
):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quant_type == "int8":
        converter.optimizations = [
            tf.lite.Optimize.DEFAULT
        ]

        converter.representative_dataset = (
            lambda: representative_data_gen(X_train_scaled)
        )

        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8
        ]

        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == "float16":
        converter.optimizations = [
            tf.lite.Optimize.DEFAULT
        ]

        converter.target_spec.supported_types = [
            tf.float16
        ]

    elif quant_type == "dynamic":
        converter.optimizations = [
            tf.lite.Optimize.DEFAULT
        ]

    else:
        raise ValueError(
            "quant_type must be 'int8', "
            "'float16', or 'dynamic'."
        )

    # Convert and save
    tflite_model = converter.convert()

    with open(filename, "wb") as file:
        file.write(tflite_model)

    # Load TFLite model
    interpreter = tf.lite.Interpreter(
        model_path=filename
    )

    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_dtype = input_details["dtype"]
    output_dtype = output_details["dtype"]

    predictions = []

    for sample in X_test:
        input_sample = sample.reshape(1, -1).astype(
            np.float32
        )

        # Quantize input when required
        if np.issubdtype(input_dtype, np.integer):
            input_scale, input_zero_point = (
                input_details["quantization"]
            )

            if input_scale == 0:
                raise ValueError(
                    "Quantized input scale cannot be zero."
                )

            input_sample = np.round(
                input_sample / input_scale
                + input_zero_point
            )

            input_limits = np.iinfo(input_dtype)

            input_sample = np.clip(
                input_sample,
                input_limits.min,
                input_limits.max,
            ).astype(input_dtype)

        else:
            input_sample = input_sample.astype(
                input_dtype
            )

        interpreter.set_tensor(
            input_details["index"],
            input_sample,
        )

        interpreter.invoke()

        output_sample = interpreter.get_tensor(
            output_details["index"]
        )[0]

        # Dequantize output when required
        if np.issubdtype(output_dtype, np.integer):
            output_scale, output_zero_point = (
                output_details["quantization"]
            )

            output_sample = (
                output_sample.astype(np.float32)
                - output_zero_point
            ) * output_scale

        prediction = int(np.argmax(output_sample))
        predictions.append(prediction)

    predictions = np.asarray(
        predictions,
        dtype=np.int32,
    )

    true_labels = np.argmax(
        y_test_cat,
        axis=1,
    )

    accuracy = float(
        np.mean(predictions == true_labels)
    )

    size_kb = file_size_kb(filename)

    print(
        f"\n{quant_type.upper()} TFLite "
        f"model size: {size_kb:.2f} KB"
    )

    print(f"Accuracy: {accuracy:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            true_labels,
            predictions,
            labels=np.arange(num_classes),
            target_names=wine.target_names,
            zero_division=0,
        )
    )

    print("Confusion matrix:")
    print(
        confusion_matrix(
            true_labels,
            predictions,
            labels=np.arange(num_classes),
        )
    )

    return {
        "filename": filename,
        "size_kb": size_kb,
        "accuracy": accuracy,
        "predictions": predictions,
    }

In [21]:
# Create and evaluate the three quantized models

int8_results = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    "int8",
    "model_int8.tflite",
)

float16_results = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    "float16",
    "model_float16.tflite",
)

dynamic_results = quantize_and_evaluate(
    model,
    X_test_scaled,
    y_test_cat,
    "dynamic",
    "model_dynamic.tflite",
)

INFO:tensorflow:Assets written to: /tmp/tmp_ovj64sq/assets


INFO:tensorflow:Assets written to: /tmp/tmp_ovj64sq/assets
/home/rambo/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-07-24 18:23:59.240658: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:23:59.240697: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:23:59.240834: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp_ovj64sq
2026-07-24 18:23:59.241623: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:23:59.241642: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp_ovj64sq
2026-07-24 18:23:59.243891: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
202


INT8 TFLite model size: 5.73 KB
Accuracy: 0.9815

Classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmpovvrd5e2/assets


INFO:tensorflow:Assets written to: /tmp/tmpovvrd5e2/assets
2026-07-24 18:23:59.682577: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:23:59.682621: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:23:59.682745: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpovvrd5e2
2026-07-24 18:23:59.683396: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:23:59.683406: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpovvrd5e2
2026-07-24 18:23:59.685070: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:23:59.707466: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpovvrd5e2
2026-07-24 18:23:59.714553: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 8.94 KB
Accuracy: 0.9815

Classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmpx2r9_gpv/assets


INFO:tensorflow:Assets written to: /tmp/tmpx2r9_gpv/assets



DYNAMIC TFLite model size: 8.16 KB
Accuracy: 0.9815

Classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.95      0.98        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


2026-07-24 18:24:00.134024: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:24:00.134066: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:24:00.134188: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpx2r9_gpv
2026-07-24 18:24:00.134911: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:24:00.134922: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpx2r9_gpv
2026-07-24 18:24:00.136863: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:24:00.161106: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpx2r9_gpv
2026-07-24 18:24:00.167372: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 33185 m

## Problem 1 - Part (c)

### Pruning

In [22]:
# Step 1: Define the pruning schedule

pruning_batch_size = 8
pruning_epochs = 10

# validation_split=0.2 means 80% is used for training
pruning_training_samples = int(
    np.floor(len(X_train_scaled) * 0.8)
)

steps_per_epoch = int(
    np.ceil(
        pruning_training_samples
        / pruning_batch_size
    )
)

end_step = steps_per_epoch * pruning_epochs

pruning_schedule = (
    tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.5,
        final_sparsity=0.7,
        begin_step=0,
        end_step=end_step,
        frequency=10,
    )
)

print("Pruning steps per epoch:", steps_per_epoch)
print("Pruning end step:", end_step)

Pruning steps per epoch: 13
Pruning end step: 130


In [23]:
# Step 2: Define the pruned model

prune_low_magnitude = (
    tfmot.sparsity.keras.prune_low_magnitude
)

pruned_model = Sequential([
    tf.keras.Input(shape=(num_features,)),

    prune_low_magnitude(
        Dense(64, activation="relu"),
        pruning_schedule=pruning_schedule,
    ),

    prune_low_magnitude(
        Dense(32, activation="relu"),
        pruning_schedule=pruning_schedule,
    ),

    prune_low_magnitude(
        Dense(
            num_classes,
            activation="softmax",
        ),
        pruning_schedule=pruning_schedule,
    ),
])

pruned_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [24]:
# Step 3: Compile and train the pruned model

pruned_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

pruning_callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

pruning_history = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=pruning_epochs,
    batch_size=pruning_batch_size,
    validation_split=0.2,
    callbacks=pruning_callbacks,
    verbose=1,
)

Epoch 1/10
13/13 [==============================] - 1s 13ms/step - loss: 1.1131 - accuracy: 0.5152 - val_loss: 0.9189 - val_accuracy: 0.7200
Epoch 2/10
13/13 [==============================] - 0s 5ms/step - loss: 0.9350 - accuracy: 0.6566 - val_loss: 0.7522 - val_accuracy: 0.9600
Epoch 3/10
13/13 [==============================] - 0s 4ms/step - loss: 0.7935 - accuracy: 0.9091 - val_loss: 0.6697 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 4ms/step - loss: 0.7010 - accuracy: 0.9495 - val_loss: 0.6037 - val_accuracy: 1.0000
Epoch 5/10
13/13 [==============================] - 0s 4ms/step - loss: 0.6201 - accuracy: 0.9495 - val_loss: 0.5263 - val_accuracy: 1.0000
Epoch 6/10
13/13 [==============================] - 0s 4ms/step - loss: 0.5245 - accuracy: 0.9697 - val_loss: 0.4369 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 5ms/step - loss: 0.4213 - accuracy: 0.9798 - val_loss: 0.3302 - val_accuracy: 1.0000
Epoch 8/10
13/13 [=

In [25]:
# Step 4: Strip pruning wrappers and convert to TFLite

stripped_pruned_model = (
    tfmot.sparsity.keras.strip_pruning(
        pruned_model
    )
)

pruned_converter = (
    tf.lite.TFLiteConverter.from_keras_model(
        stripped_pruned_model
    )
)

pruned_tflite_model = pruned_converter.convert()

with open("model_pruned.tflite", "wb") as file:
    file.write(pruned_tflite_model)

pruned_tflite_size_kb = file_size_kb(
    "model_pruned.tflite"
)

print(
    f"Pruned TFLite model size: "
    f"{pruned_tflite_size_kb:.2f} KB"
)

INFO:tensorflow:Assets written to: /tmp/tmp47f2yldy/assets


INFO:tensorflow:Assets written to: /tmp/tmp47f2yldy/assets


Pruned TFLite model size: 14.09 KB


2026-07-24 18:24:14.300243: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:24:14.300294: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:24:14.300414: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp47f2yldy
2026-07-24 18:24:14.300782: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:24:14.300789: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp47f2yldy
2026-07-24 18:24:14.301541: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:24:14.312741: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp47f2yldy
2026-07-24 18:24:14.316580: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 16166 m

In [26]:
# Step 5: Evaluate the stripped pruned model

pruned_probabilities = (
    stripped_pruned_model.predict(
        X_test_scaled,
        verbose=0,
    )
)

pruned_predictions = np.argmax(
    pruned_probabilities,
    axis=1,
)

pruned_accuracy = float(
    np.mean(pruned_predictions == y_test)
)

print(
    f"Pruned-model accuracy: "
    f"{pruned_accuracy:.4f}"
)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        pruned_predictions,
        labels=np.arange(num_classes),
        target_names=wine.target_names,
        zero_division=0,
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_test,
        pruned_predictions,
        labels=np.arange(num_classes),
    )
)

Pruned-model accuracy: 0.9815

Classification report:
              precision    recall  f1-score   support

     class_0       0.95      1.00      0.97        18
     class_1       1.00      0.95      0.98        21
     class_2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [27]:
# Step 1: Define the student model

student_model = Sequential([
    tf.keras.Input(shape=(num_features,)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(num_classes, activation="softmax"),
])

student_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [28]:
# Step 2: Generate teacher soft labels

teacher_preds_soft = model.predict(
    X_train_scaled,
    verbose=0,
)

print(
    "Teacher soft-label shape:",
    teacher_preds_soft.shape,
)

Teacher soft-label shape: (124, 3)


In [29]:
# Step 3: Combine hard and soft labels

combined_train_labels = np.concatenate(
    [
        y_train_cat,
        teacher_preds_soft,
    ],
    axis=1,
).astype(np.float32)

alpha = 0.5


def distillation_loss(
    y_true_combined,
    y_pred,
):
    y_true_hard = (
        y_true_combined[:, :num_classes]
    )

    y_true_soft = (
        y_true_combined[:, num_classes:]
    )

    hard_loss = (
        tf.keras.losses.categorical_crossentropy(
            y_true_hard,
            y_pred,
        )
    )

    soft_loss = (
        tf.keras.losses.categorical_crossentropy(
            y_true_soft,
            y_pred,
        )
    )

    return (
        alpha * hard_loss
        + (1.0 - alpha) * soft_loss
    )


def hard_label_accuracy(
    y_true_combined,
    y_pred,
):
    y_true_hard = (
        y_true_combined[:, :num_classes]
    )

    return tf.keras.metrics.categorical_accuracy(
        y_true_hard,
        y_pred,
    )


print(
    "Combined-label shape:",
    combined_train_labels.shape,
)

Combined-label shape: (124, 6)


In [30]:
# Step 4: Compile and train the student model

student_model.compile(
    optimizer="adam",
    loss=distillation_loss,
    metrics=[hard_label_accuracy],
)

student_history = student_model.fit(
    X_train_scaled,
    combined_train_labels,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1,
)

Epoch 1/10
13/13 [==============================] - 1s 12ms/step - loss: 1.1003 - hard_label_accuracy: 0.4848 - val_loss: 0.8870 - val_hard_label_accuracy: 0.6000
Epoch 2/10
13/13 [==============================] - 0s 4ms/step - loss: 0.9523 - hard_label_accuracy: 0.5960 - val_loss: 0.7817 - val_hard_label_accuracy: 0.7200
Epoch 3/10
13/13 [==============================] - 0s 4ms/step - loss: 0.8321 - hard_label_accuracy: 0.7071 - val_loss: 0.7051 - val_hard_label_accuracy: 0.8000
Epoch 4/10
13/13 [==============================] - 0s 3ms/step - loss: 0.7298 - hard_label_accuracy: 0.8384 - val_loss: 0.6320 - val_hard_label_accuracy: 0.8000
Epoch 5/10
13/13 [==============================] - 0s 4ms/step - loss: 0.6356 - hard_label_accuracy: 0.9091 - val_loss: 0.5576 - val_hard_label_accuracy: 0.8800
Epoch 6/10
13/13 [==============================] - 0s 3ms/step - loss: 0.5398 - hard_label_accuracy: 0.9394 - val_loss: 0.4829 - val_hard_label_accuracy: 0.9200
Epoch 7/10
13/13 [=========

In [31]:
# Step 5: Convert the student model to TFLite

kd_converter = (
    tf.lite.TFLiteConverter.from_keras_model(
        student_model
    )
)

kd_tflite_model = kd_converter.convert()

with open("model_kd.tflite", "wb") as file:
    file.write(kd_tflite_model)

kd_tflite_size_kb = file_size_kb(
    "model_kd.tflite"
)

print(
    f"Knowledge-distilled TFLite model size: "
    f"{kd_tflite_size_kb:.2f} KB"
)

INFO:tensorflow:Assets written to: /tmp/tmpzup7ulo1/assets


INFO:tensorflow:Assets written to: /tmp/tmpzup7ulo1/assets


Knowledge-distilled TFLite model size: 6.09 KB


2026-07-24 18:24:28.178360: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:24:28.178409: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:24:28.178533: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpzup7ulo1
2026-07-24 18:24:28.179290: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:24:28.179301: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpzup7ulo1
2026-07-24 18:24:28.181324: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:24:28.205589: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpzup7ulo1
2026-07-24 18:24:28.212277: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 33743 m

In [32]:
# Step 6: Evaluate the student model

student_probabilities = student_model.predict(
    X_test_scaled,
    verbose=0,
)

student_predictions = np.argmax(
    student_probabilities,
    axis=1,
)

student_accuracy = float(
    np.mean(student_predictions == y_test)
)

print(
    f"Student-model accuracy: "
    f"{student_accuracy:.4f}"
)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        student_predictions,
        labels=np.arange(num_classes),
        target_names=wine.target_names,
        zero_division=0,
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_test,
        student_predictions,
        labels=np.arange(num_classes),
    )
)

Student-model accuracy: 0.9630

Classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        18
     class_1       1.00      0.90      0.95        21
     class_2       0.88      1.00      0.94        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion matrix:
[[18  0  0]
 [ 0 19  2]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [33]:
# Tiny linear student model

tiny_student_model = Sequential([
    tf.keras.Input(shape=(num_features,)),
    Dense(
        num_classes,
        activation="softmax",
    ),
])

tiny_student_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.01
    ),
    loss=distillation_loss,
    metrics=[hard_label_accuracy],
)

tiny_student_history = tiny_student_model.fit(
    X_train_scaled,
    combined_train_labels,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=0,
)

# Evaluate the unquantized tiny student

tiny_keras_probabilities = (
    tiny_student_model.predict(
        X_test_scaled,
        verbose=0,
    )
)

tiny_keras_predictions = np.argmax(
    tiny_keras_probabilities,
    axis=1,
)

tiny_keras_accuracy = float(
    np.mean(
        tiny_keras_predictions == y_test
    )
)

print(
    f"Tiny student Keras accuracy: "
    f"{tiny_keras_accuracy:.4f}"
)

# Apply full int8 quantization

tiny_int8_results = quantize_and_evaluate(
    tiny_student_model,
    X_test_scaled,
    y_test_cat,
    "int8",
    "model_tiny_kd_int8.tflite",
)

# Compare all models

comparison = pd.DataFrame([
    {
        "Model": "Base",
        "Size (KB)": base_tflite_size_kb,
        "Accuracy": base_test_accuracy,
    },
    {
        "Model": "Base int8",
        "Size (KB)": int8_results["size_kb"],
        "Accuracy": int8_results["accuracy"],
    },
    {
        "Model": "Base float16",
        "Size (KB)": float16_results["size_kb"],
        "Accuracy": float16_results["accuracy"],
    },
    {
        "Model": "Base dynamic",
        "Size (KB)": dynamic_results["size_kb"],
        "Accuracy": dynamic_results["accuracy"],
    },
    {
        "Model": "Pruned",
        "Size (KB)": pruned_tflite_size_kb,
        "Accuracy": pruned_accuracy,
    },
    {
        "Model": "Knowledge-distilled",
        "Size (KB)": kd_tflite_size_kb,
        "Accuracy": student_accuracy,
    },
    {
        "Model": "Tiny KD + int8",
        "Size (KB)": (
            tiny_int8_results["size_kb"]
        ),
        "Accuracy": (
            tiny_int8_results["accuracy"]
        ),
    },
])

comparison = comparison.sort_values(
    "Size (KB)"
).reset_index(drop=True)

print("\nModel comparison, smallest first:")

print(
    comparison.to_string(
        index=False,
        formatters={
            "Size (KB)": (
                lambda value: f"{value:.2f}"
            ),
            "Accuracy": (
                lambda value: f"{value:.4f}"
            ),
        },
    )
)

# Calculate improvement over the previous smallest model

previous_smallest_size = min(
    int8_results["size_kb"],
    float16_results["size_kb"],
    dynamic_results["size_kb"],
    pruned_tflite_size_kb,
    kd_tflite_size_kb,
)

size_reduction_percent = (
    100.0
    * (
        previous_smallest_size
        - tiny_int8_results["size_kb"]
    )
    / previous_smallest_size
)

accuracy_change = (
    tiny_int8_results["accuracy"]
    - base_test_accuracy
)

print(
    "\nSize reduction relative to the "
    "smallest model from parts (b)-(d): "
    f"{size_reduction_percent:.2f}%"
)

print(
    "Accuracy change relative to the "
    f"base model: {accuracy_change:+.4f}"
)

Tiny student Keras accuracy: 0.9630
INFO:tensorflow:Assets written to: /tmp/tmpecdzmnm_/assets


INFO:tensorflow:Assets written to: /tmp/tmpecdzmnm_/assets



INT8 TFLite model size: 1.49 KB
Accuracy: 0.9630

Classification report:
              precision    recall  f1-score   support

     class_0       0.95      1.00      0.97        18
     class_1       1.00      0.90      0.95        21
     class_2       0.94      1.00      0.97        15

    accuracy                           0.96        54
   macro avg       0.96      0.97      0.96        54
weighted avg       0.97      0.96      0.96        54

Confusion matrix:
[[18  0  0]
 [ 1 19  1]
 [ 0  0 15]]

Model comparison, smallest first:
              Model Size (KB) Accuracy
     Tiny KD + int8      1.49   0.9630
          Base int8      5.73   0.9815
Knowledge-distilled      6.09   0.9630
       Base dynamic      8.16   0.9815
       Base float16      8.94   0.9815
               Base     14.06   0.9815
             Pruned     14.09   0.9815

Size reduction relative to the smallest model from parts (b)-(d): 73.98%
Accuracy change relative to the base model: -0.0185


/home/rambo/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-07-24 18:24:41.374515: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:24:41.374556: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:24:41.374672: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpecdzmnm_
2026-07-24 18:24:41.374962: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:24:41.374968: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpecdzmnm_
2026-07-24 18:24:41.375738: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:24:41.389494: I tensorflow/cc/saved_model/loader

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
